In [1]:
import requests
import re
import time

RIOT_API_KEY = "RGAPI-07231d99-4076-4785-850a-79d1332ac1cc"  # ← 填你的key
REGIONAL_ROUTING = "americas"  # americas / europe / asia

HEADERS = {"X-Riot-Token": RIOT_API_KEY}

In [11]:
def parse_match_id(match_id):
    m = re.match(r"([A-Z0-9]+)_(\d+)", match_id)
    if not m:
        raise ValueError("Invalid matchId")
    return m.group(1), int(m.group(2))


def get_match_queue(match_id):
    """
    返回:
    - queueId (int) 如果成功
    - None 如果404或错误
    """
    url = f"https://{REGIONAL_ROUTING}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    r = requests.get(url, headers=HEADERS)

    if r.status_code == 200:
        data = r.json()
        return data["info"]["queueId"]

    elif r.status_code == 404:
        return None

    elif r.status_code == 429:
        print("Rate limited — sleeping 2s")
        time.sleep(2)
        return get_match_queue(match_id)

    else:
        return None

In [19]:
def test_next_match_solo_duo(match_id, loops=1000, sleep=0.3):
    platform, num = parse_match_id(match_id)

    success = 0
    not_solo = 0
    not_found = 0

    print(f"Starting from {match_id}")
    print("Loops:", loops)
    print("-"*40)

    for i in range(loops):
        next_match_id = f"{platform}_{num + i + 1}"

        queue = get_match_queue(next_match_id)

        if queue == 420:
            success += 1
            print(next_match_id)
        elif queue is None:
            not_found += 1
        else:
            not_solo += 1

        if (i+1) % 50 == 0:
            print(f"Checked {i+1}")

        time.sleep(sleep)

    print("\n===== RESULT =====")
    print("Solo/Duo success:", success)
    print("Other queue:", not_solo)
    print("Not found:", not_found)
    print("Solo rate:", success / loops)

    return success, not_solo, not_found

In [20]:
match_id = "NA1_5489311424"

test_next_match_solo_duo(match_id, loops=100)

Starting from NA1_5489311424
Loops: 100
----------------------------------------
NA1_5489311453
NA1_5489311469
Checked 50
NA1_5489311485
NA1_5489311496
NA1_5489311497
NA1_5489311500
NA1_5489311502
NA1_5489311503
NA1_5489311514
NA1_5489311523
Checked 100

===== RESULT =====
Solo/Duo success: 10
Other queue: 10
Not found: 80
Solo rate: 0.1


(10, 10, 80)